# 2022 Victorian state election: lower house results

Seat winners from the VEC final distribution of preferences (`data/vec_2022_two_candidate_preferred.csv`, scraped with `utils.vec_results.scrape_two_candidate_preferred`). Narracan is the January 2023 supplementary election.

In [1]:
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go

from utils.spatial import load_2022_districts
from utils.vec_results import GROUPS, load_first_preference_shares, load_two_candidate_preferred

PARTY_COLOURS = {
    "Labor": "red",
    "L/NP": "blue",
    "Greens": "green",
    "One Nation": "orange",
    "Others": "grey",
}

gdf = load_2022_districts().to_crs(4326)

## simplified geojson for plotly so the maps don't get huge
geo = gdf.set_index("DISTRICT")[["geometry"]].copy()
geo["geometry"] = geo.simplify(0.001)
geojson = geo.__geo_interface__

res = load_two_candidate_preferred().join(load_first_preference_shares()[GROUPS])
assert res.index.sort_values().equals(geo.index.sort_values()), "district names don't line up"
res["winner_party"].value_counts()

winner_party
Australian Labor Party - Victorian Branch    56
Liberal                                      19
The Nationals                                 9
Australian Greens                             4
Name: count, dtype: int64

## Seat winners

Each district is coloured by the winning party. In most seats the final count is the 2CP; in a few the VEC stopped once someone passed 50%, so it's 3+ candidates. Hover for the result and first preferences.

In [2]:
hover = (
    res["winner"] + " (" + res["winner_party"] + ")<br>"
    + "Final count: " + res["winner_group"] + " " + res["winner_share"].map("{:.1%}".format)
    + " v " + res["runner_up_group"] + " " + res["runner_up_share"].map("{:.1%}".format) + "<br><br>"
    + "First preferences:<br>"
    + res[GROUPS].apply(lambda r: "<br>".join(f"{p}: {r[p]:.1%}" for p in GROUPS), axis=1)
)

counts = res["winner_group"].value_counts()

fig = go.Figure()
for p, colour in PARTY_COLOURS.items():
    d = res[res["winner_group"] == p]
    if d.empty:
        continue
    fig.add_trace(go.Choroplethmap(
        geojson=geojson,
        locations=d.index,
        z=[1] * len(d),                       # constant: solid party colour
        colorscale=[[0, colour], [1, colour]],
        showscale=False,
        showlegend=False,
        legendgroup=p,          # clicking the legend entry toggles this trace
        marker_opacity=0.8,
        marker_line_width=0.5,
        marker_line_color="white",
        name=p,
        text=d.index + "<br>" + hover[d.index],
        hovertemplate="<b>%{text}</b><extra></extra>",
    ))
    # dummy trace that only exists to draw the legend swatch
    fig.add_trace(go.Scattermap(
        lat=[None], lon=[None],
        mode="markers",
        marker=dict(size=12, color=colour),
        name=f"{p} ({counts[p]})",
        legendgroup=p,
        showlegend=True,
        hoverinfo="skip",
    ))

fig.update_layout(
    map_style="carto-positron",
    map_zoom=5.6,
    map_center={"lat": -37.0, "lon": 145.0},
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
    title="2022 Victorian state election: seat winners (Legislative Assembly)",
    legend=dict(
        title="Winning party (seats)",
        x=0.01, y=0.99,
        bgcolor="rgba(255,255,255,0.8)",
    ),
)
fig.show()

In [3]:
## closest seats on the final count
res.sort_values("winner_share")[
    ["winner", "winner_party", "winner_share", "runner_up_group", "runner_up_share"]
].head(15)

,winner,winner_party,winner_share,runner_up_group,runner_up_share
district,,,,,
CARRUM,"KILKENNY, Sonya",Australian Labor Party - Victorian Branch,0.500370,L/NP,0.305935
BENDIGO EAST,"ALLAN, Jacinta",Australian Labor Party - Victorian Branch,0.500395,L/NP,0.295163
BERWICK,"BATTIN, Brad",Liberal,0.501073,Labor,0.372621
YAN YEAN,"KATHAGE, Lauren",Australian Labor Party - Victorian Branch,0.501560,L/NP,0.382024
NORTHCOTE,"THEOPHANOUS, Kat",Australian Labor Party - Victorian Branch,0.502157,Greens,0.497843
BASS,"CRUGNALE, Jordan",Australian Labor Party - Victorian Branch,0.502439,L/NP,0.497561
LARA,"GEORGE, Ella",Australian Labor Party - Victorian Branch,0.503074,L/NP,0.218643
BRIGHTON,"NEWBURY, James",Liberal,0.503204,Labor,0.287649
BUNDOORA,"BROOKS, Colin",Australian Labor Party - Victorian Branch,0.503669,L/NP,0.329631
